# Basic

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import polars as pl
import networkx as nx


import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.graph_fct as graph_fct
import src.graph_tokenizer_gd_tree_dev.utils as utils



In [3]:
id_to_label, combined_subgraphs = graph_fct.get_combined_combined_subgraphs_and_id2label()
df_mapped = pl.read_parquet(f"{config.BasicConfig().mapped_path}")
mapped_ids = df_mapped["id"].unique().to_list()

Ks = config.TokenizerParam().Ks
rnd_iters = config.TokenizerParam().rnd_iters
D = nx.dag_longest_path_length(combined_subgraphs)   # returns the length (number of edges) of the longest path

# k random

In [8]:
nodes_df = (pl.DataFrame(
    [{"token": n, **attrs} for n, attrs in combined_subgraphs.nodes(data=True)])
    .with_columns(
        pl.col("token").replace_strict(id_to_label, default=None).alias("label"))
)


In [9]:
frames = []
for k in Ks:
    for it in config.TokenizerParam().rnd_iters:
        df_sampled = (
            nodes_df.sample(n=k)
            .with_columns(
                pl.lit(k).alias("k"),
                pl.lit(it).alias("iter"))
            .with_row_index()
        )

        frames.append(df_sampled)

df_all = pl.concat(frames)
df_all.write_parquet(config.CandidateLists().k_random_all_samples)

In [10]:
df_all

index,token,label,k,iter
u32,str,str,i32,i32
0,"""17636008""","""Specimen collection (procedure…",500,0
1,"""278292003""","""Ultrasound imaging - action (q…",500,0
2,"""276341003""","""Cardiovascular investigation (…",500,0
3,"""72670004""","""Sign (finding)""",500,0
4,"""412553001""","""Rifaximin (substance)""",500,0
…,…,…,…,…
19495,"""36432003+86481000""","""36432003|Anastomosis of urinar…",19500,19
19496,"""14012001""","""Common law partnership (findin…",19500,19
19497,"""33156000:{405813007=(245183002…","""33156000|Incision of tendon (p…",19500,19


# highest degree and is_a child dataframe construction

In [11]:
# all relationship distances in the combined subgraph
all_pairs = nx.all_pairs_shortest_path_length(combined_subgraphs)

distance_df = pl.DataFrame(
    [
        {"src_id": src, "dst_id": dst, "distance": dist}
        for src, lengths in all_pairs
        for dst, dist in lengths.items()
        if src != dst
    ]
)
self_distance_df = pl.DataFrame(
    [
        {"src_id": n, "dst_id": n, "distance": 0}
        for n in combined_subgraphs.nodes()
    ]
)

distance_df = pl.concat([distance_df, self_distance_df])

distance_df.write_parquet(config.ProcessedGraph().all_rel_distance_subgraph)


In [12]:
# is_a only distances in the combined subgraph
is_a_edges = [
    (u, v, k) for u, v, k, attrs in combined_subgraphs.edges(keys=True, data=True)
    if attrs.get("relation") == "IS_A"
]
is_a_subgraph = nx.MultiDiGraph(combined_subgraphs.edge_subgraph(is_a_edges))  # materialize, don't keep it a view

all_pairs = nx.all_pairs_shortest_path_length(is_a_subgraph)
distance_df = pl.DataFrame(
    [
        {"src_id": src, "dst_id": dst, "distance": dist}
        for src, lengths in all_pairs
        for dst, dist in lengths.items()
        if src != dst
    ]
)
self_distance_df = pl.DataFrame(
    [
        {"src_id": n, "dst_id": n, "distance": 0}
        for n in combined_subgraphs.nodes()
    ]
)
distance_df = pl.concat([distance_df, self_distance_df])
distance_df.write_parquet(config.ProcessedGraph().is_a_distance_subgraph)

# highest degree candidate selection

In [15]:
(pl.read_parquet(config.ProcessedGraph().all_rel_distance_subgraph)
# .filter(pl.col("distance") <= config.TokenizerParam().max_dist_candidate)
.group_by("dst_id")
.agg(pl.col("src_id").n_unique().alias("num_in_edges"))
.sort(by = "num_in_edges", descending = True)
.with_row_index()
.rename({"dst_id": "token"})
.write_parquet(config.CandidateLists().highest_degree)
)

In [16]:
pl.read_parquet(config.CandidateLists().highest_degree)

index,token,num_in_edges
u32,str,u32
0,"""362981000""",52943
1,"""123037004""",44573
2,"""442083009""",41216
3,"""91723000""",40923
4,"""71388002""",33871
…,…,…
75959,"""117033003:116686009=258442002,…",1
75960,"""133921002:{246454002=255398004…",1
75961,"""363871006+782487009:{704321009…",1


# highest degree candidate selection dist <= 1

In [17]:
MAX_DIST = 1
(pl.read_parquet(config.ProcessedGraph().all_rel_distance_subgraph)
.filter(pl.col("distance") <= MAX_DIST)
.group_by("dst_id")
.agg(pl.col("src_id").n_unique().alias("num_in_edges"))
.sort(by = "num_in_edges", descending = True)
.with_row_index()
.rename({"dst_id": "token"})
.write_parquet(config.CandidateLists().highest_degree_dist_1)
)

# most children via is_a

In [18]:
(pl.read_parquet(config.ProcessedGraph().is_a_distance_subgraph)
.group_by("dst_id")
.agg(pl.col("src_id").n_unique().alias("num_in_edges"))
.sort(by = "num_in_edges", descending = True)
.with_row_index()
.rename({"dst_id": "token"})
.write_parquet(config.CandidateLists().most_children)
 )

# page rank

In [21]:
pagerank = nx.pagerank(combined_subgraphs)
df_pagerank = utils.to_ranked_df(id_to_label, pagerank, "pagerank")
df_pagerank.write_parquet(config.CandidateLists().pagerank)
df_pagerank.head()

index,token,pagerank,label
u32,str,f64,str
0,"""362981000""",0.034617,"""Qualifier value (qualifier val…"
1,"""129264002""",0.020216,"""Action (qualifier value)"""
2,"""129265001""",0.016587,"""Evaluation - action (qualifier…"
3,"""123037004""",0.016474,"""Body structure (body structure…"
4,"""91723000""",0.01415,"""Anatomical structure (body str…"


## Personalized PageRank seeded on `M`

Same call, but the random walk restarts at a mapped concept instead of a uniformly random
node -- `personalization={m: 1 for m in mapped_ids}` (networkx normalizes this internally;
nodes absent from the dict get personalization value 0).

In [23]:
personalized_pagerank = nx.pagerank(
    combined_subgraphs,
    personalization={m: 1 for m in mapped_ids if m in combined_subgraphs},
)
df_ppr = utils.to_ranked_df(id_to_label, personalized_pagerank, "personalized_pagerank")
df_ppr.write_parquet(config.CandidateLists().personalized_pagerank)
df_ppr.head()

index,token,personalized_pagerank,label
u32,str,f64,str
0,"""362981000""",0.036489,"""Qualifier value (qualifier val…"
1,"""129264002""",0.025688,"""Action (qualifier value)"""
2,"""129265001""",0.023401,"""Evaluation - action (qualifier…"
3,"""123037004""",0.015489,"""Body structure (body structure…"
4,"""91723000""",0.012761,"""Anatomical structure (body str…"


# closeness

In [24]:
G_simple = nx.DiGraph(combined_subgraphs)  # multi-edges collapsed, shared by both cells below

closeness_centrality = nx.closeness_centrality(G_simple)
df_closeness = utils.to_ranked_df(id_to_label, closeness_centrality, "closeness_centrality")
df_closeness.write_parquet(config.CandidateLists().closeness_centrality)
df_closeness.head()

index,token,closeness_centrality,label
u32,str,f64,str
0,"""129265001""",0.187539,"""Evaluation - action (qualifier…"
1,"""362981000""",0.173651,"""Qualifier value (qualifier val…"
2,"""129264002""",0.151543,"""Action (qualifier value)"""
3,"""91723000""",0.106717,"""Anatomical structure (body str…"
4,"""442083009""",0.093629,"""Anatomical or acquired body st…"


# discrete set

In [5]:
import src.graph_tokenizer_gd_tree_dev.classical_selectors as classical_selectors

distance_df = pl.read_parquet(config.ProcessedGraph().all_rel_distance_subgraph)

set_cover_selector = classical_selectors.HardGreedySetCoverSelector(distance_df, mapped_ids, D)
history_set_cover = set_cover_selector.select(k=len(mapped_ids), verbose=True, progress_every=5000)

print(f"\nranked {len(history_set_cover):,} candidates "
      f"(of {len(set_cover_selector.covers):,} that cover >=1 mapped concept; "
      f"{len(set_cover_selector._leftover):,} zero-coverage candidates padded at the tail)")

df_set_cover = (
    pl.DataFrame(history_set_cover, schema=["token", "gain", "cumulative_score"], orient="row")
    .with_columns(pl.col("token").replace_strict(id_to_label, default=None).alias("label"))
    .with_row_index()
)
df_set_cover.write_parquet(config.CandidateLists().discrete_set_cover)
df_set_cover.head()

[  5000/46150] token='126912001'     gain=   0  covered=46146/46150 (99.991%)  (re-evals=1)
[ 10000/46150] token='230286002'     gain=   0  covered=46146/46150 (99.991%)  (re-evals=1)
[ 15000/46150] token='281704007'     gain=   0  covered=46146/46150 (99.991%)  (re-evals=1)
[ 20000/46150] token='364494008'     gain=   0  covered=46146/46150 (99.991%)  (re-evals=1)
[ 25000/46150] token='418248000'     gain=   0  covered=46146/46150 (99.991%)  (re-evals=1)
[ 30000/46150] token='608852006'     gain=   0  covered=46146/46150 (99.991%)  (re-evals=1)
[ 35000/46150] token='77068002'      gain=   0  covered=46146/46150 (99.991%)  (re-evals=1)

ranked 39,078 candidates (of 39,078 that cover >=1 mapped concept; 0 zero-coverage candidates padded at the tail)


index,token,gain,cumulative_score,label
u32,str,i64,f64,str
0,"""362981000""",36944,0.80052,"""Qualifier value (qualifier val…"
1,"""404684003""",5639,0.922709,"""Clinical finding (finding)"""
2,"""363787002""",1177,0.948212,"""Observable entity (observable …"
3,"""105590001""",666,0.962644,"""Substance (substance)"""
4,"""123037004""",481,0.973066,"""Body structure (body structure…"
